In [1]:
# catboostとoptunaのインストール
!pip install catboost optuna

# 10年定着予測 - GBDT Ensemble with Advanced Feature Engineering

**目的**: 上位3モデル（CatBoost, LightGBM, XGBoost）のアンサンブルと、さらに高度な特徴量エンジニアリングを実装する。

## 実装する特徴量

### 既存特徴量（04_baseline_lgbm_feature_engineeringから）
1. 月次データの集約特徴量（統計量、時期別、トレンド）
2. 月次カテゴリカル変数の変化パターン
3. 欠損値特徴量
4. ドメイン知識特徴量
5. テキスト特徴量
6. 時間的特徴量
7. 交互作用特徴量

### 新規追加：高度な特徴量
8. **多項式特徴量** - 重要特徴量の2次/3次項
9. **統計的特徴量** - 歪度、尖度、パーセンタイル
10. **ビン化特徴量** - 連続値の離散化
11. **クラスター特徴量** - KMeansクラスタリング
12. **複雑な交互作用** - 3変数以上の組み合わせ
13. **時系列ラグ特徴量** - 経過月数ごとの差分
14. **比率・割合特徴量** - 各種指標の相対値

## モデリング戦略

### 個別モデル学習
- **CatBoost** (catboost_model_v2.py)
- **LightGBM** (lgbm_model_v2.py)
- **XGBoost** (xgboost_model_v2.py)
- TimeSeriesSplit（入社日順）を使用

### アンサンブル手法
1. **Simple Average** - 3モデルの単純平均
2. **Weighted Average** - OOFスコアベースの重み付き平均
3. **Stacking** - メタモデル（Logistic Regression）

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Aug  8 08:05:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import psutil



# メモリ情報を取得
mem = psutil.virtual_memory()
total_memory_gb = mem.total / (1024**3) # バイトをギガバイトに変換
available_memory_gb = mem.available / (1024**3)
used_memory_gb = mem.used / (1024**3)

print(f"総メモリ: {total_memory_gb:.2f} GB")
print(f"利用可能メモリ: {available_memory_gb:.2f} GB")
print(f"使用済みメモリ: {used_memory_gb:.2f} GB")

# より詳細なメモリ情報 (linux コマンド)
!cat /proc/meminfo

import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# プロジェクトルートの設定（Google Drive）
PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026") # Google Drive内の正しいパスに修正
sys.path.append(str(PROJECT_ROOT))

総メモリ: 50.99 GB
利用可能メモリ: 49.48 GB
使用済みメモリ: 0.91 GB
MemTotal:       53467192 kB
MemFree:        45929988 kB
MemAvailable:   51879052 kB
Buffers:          186888 kB
Cached:          6118952 kB
SwapCached:            0 kB
Active:          1192444 kB
Inactive:        5720280 kB
Active(anon):       1536 kB
Inactive(anon):   607624 kB
Active(file):    1190908 kB
Inactive(file):  5112656 kB
Unevictable:          20 kB
Mlocked:              20 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:               520 kB
Writeback:             0 kB
AnonPages:        606940 kB
Mapped:           500128 kB
Shmem:              2264 kB
KReclaimable:     274440 kB
Slab:             345352 kB
SReclaimable:     274440 kB
SUnreclaim:        70912 kB
KernelStack:        8528 kB
PageTables:         8920 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:    26733596 kB
Committed_AS:    4226596 kB
VmallocTotal:   3435973836

In [4]:
import datetime
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit

!pip install catboost

# モジュールのインポート
from common.lgbm.lgbm_model_v2 import run_lgb
from common.xgboost.xgb_model_v2 import run_xgb
from common.catboost.cat_model_v2 import run_cat
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")

# 乱数シードの固定
SEED = 42
seed_everything(seed=SEED)

# ターゲット列とID列の設定
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

# 表示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# プロット設定
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [5]:
import datetime
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit

# モジュールのインポート
from common.lgbm.lgbm_model_v2 import run_lgb
from common.xgboost.xgb_model_v2 import run_xgb
from common.catboost.cat_model_v2 import run_cat
from common.lgbm.lgbm_model_optuna_v2 import run_lgb_optuna
from common.xgboost.xgb_model_optuna_v2 import run_xgb_optuna
from common.catboost.cat_model_optuna_v2 import run_cat_optuna
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")

# 乱数シードの固定
SEED = 42
seed_everything(seed=SEED)

# ターゲット列とID列の設定
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

# 表示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# プロット設定
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [6]:
# スクリプト名・日付・保存パスの設定
SCRIPT_NAME = "11_ensemble_gbdt_advanced_features"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# ログディレクトリ
LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

# 出力ディレクトリ
OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 保存パス
SUBMISSION_SIMPLE_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_simple_avg.csv"
SUBMISSION_WEIGHTED_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_weighted_avg.csv"
SUBMISSION_STACKING_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_stacking.csv"

# モデル保存ディレクトリ
SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Saved Models Directory: {SAVED_MODELS_DIR}")

[2026-08-08 08:05:20] [INFO] === [11_ensemble_gbdt_advanced_features] 実験開始 ===


INFO:11_ensemble_gbdt_advanced_features:=== [11_ensemble_gbdt_advanced_features] 実験開始 ===


[2026-08-08 08:05:20] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


INFO:11_ensemble_gbdt_advanced_features:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


[2026-08-08 08:05:20] [INFO] Saved Models Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/11_ensemble_gbdt_advanced_features


INFO:11_ensemble_gbdt_advanced_features:Saved Models Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/11_ensemble_gbdt_advanced_features


In [7]:
# データの読み込み
INPUT_DIR = PROJECT_ROOT / "data" / "input"

# 属性データの読み込み (社員1名 = 1行)
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")

# 月次データの読み込み (社員1名 × 24か月 = 複数行)
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

# ターゲット変数の取得
y_train = train_persona[TARGET_COL]

# 社員IDリスト
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-08 08:05:21] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:11_ensemble_gbdt_advanced_features:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-08 08:05:21] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:11_ensemble_gbdt_advanced_features:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-08 08:05:21] [INFO] 定着率: 0.5647


INFO:11_ensemble_gbdt_advanced_features:定着率: 0.5647


[2026-08-08 08:05:21] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:11_ensemble_gbdt_advanced_features:Train IDs: 2761, Test IDs: 2502


## 1. 月次データの集約特徴量（既存）

0-23ヶ月の月次データから、以下の特徴量を生成します：

- **統計量ベース**: mean, std, min, max, median, cv
- **時期別統計量**: 初期3ヶ月、中期、後期、差分
- **トレンド特徴量**: 線形回帰の傾き、差、比率

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """
    月次データから集約特徴量を生成
    """
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            # 統計量
            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            # 時期別
            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()

            # トレンド
            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次集約特徴量関数定義完了")

✅ 月次集約特徴量関数定義完了


In [9]:
def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)

def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次カテゴリ変化+欠損値+ドメイン特徴量関数定義完了")

✅ 月次カテゴリ変化+欠損値+ドメイン特徴量関数定義完了


## 2. 新規：高度な特徴量エンジニアリング

以下の高度な特徴量を追加してモデル性能を向上させます：

- **統計的特徴量**: 歪度、尖度、パーセンタイル
- **多項式特徴量**: 重要特徴量の2次項
- **比率・割合特徴量**: 各種指標の相対値
- **クラスター特徴量**: 月次データのKMeansクラスタリング
- **複雑な交互作用**: 3変数以上の組み合わせ

In [10]:
def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

def create_cluster_features(monthly_df, employee_ids, n_clusters=5):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]

print("✅ 高度な特徴量関数定義完了")

✅ 高度な特徴量関数定義完了


## 3. 特徴量生成の実行

定義した関数を使用して、Train/Testデータの特徴量を生成します。

In [11]:
logger.info("-" * 60)
logger.info("特徴量生成開始")
logger.info("-" * 60)

logger.info("月次集約特徴量を生成中...")
train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)
logger.info(f"Train: {train_monthly_agg.shape}, Test: {test_monthly_agg.shape}")

logger.info("月次カテゴリ変化特徴量を生成中...")
train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)
logger.info(f"Train: {train_cat_change.shape}, Test: {test_cat_change.shape}")

logger.info("欠損値特徴量を生成中...")
train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)
logger.info(f"Train: {train_missing.shape}, Test: {test_missing.shape}")

logger.info("ドメイン知識特徴量を生成中...")
train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)
logger.info(f"Train: {train_domain.shape}, Test: {test_domain.shape}")

logger.info("高度な統計特徴量を生成中...")
train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)
logger.info(f"Train: {train_advanced_stats.shape}, Test: {test_advanced_stats.shape}")

logger.info("クラスター特徴量を生成中...")
train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5)
logger.info(f"Train: {train_cluster.shape}, Test: {test_cluster.shape}")

[2026-08-08 08:05:22] [INFO] ------------------------------------------------------------


INFO:11_ensemble_gbdt_advanced_features:------------------------------------------------------------


[2026-08-08 08:05:22] [INFO] 特徴量生成開始


INFO:11_ensemble_gbdt_advanced_features:特徴量生成開始


[2026-08-08 08:05:22] [INFO] ------------------------------------------------------------


...

INFO:11_ensemble_gbdt_advanced_features:Train: (2761, 26), Test: (2502, 26)


[2026-08-08 08:11:04] [INFO] クラスター特徴量を生成中...


INFO:11_ensemble_gbdt_advanced_features:クラスター特徴量を生成中...


[2026-08-08 08:11:32] [INFO] Train: (2761, 2), Test: (2502, 2)


INFO:11_ensemble_gbdt_advanced_features:Train: (2761, 2), Test: (2502, 2)


In [12]:
logger.info("テキスト特徴量を生成中...")
text_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
train_persona["text_total_chars"] = train_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
for col in text_cols:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)

logger.info("時間的特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

logger.info("交互作用特徴量を生成中...")
train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

logger.info("特徴量処理完了")

[2026-08-08 08:11:32] [INFO] テキスト特徴量を生成中...


INFO:11_ensemble_gbdt_advanced_features:テキスト特徴量を生成中...


[2026-08-08 08:11:32] [INFO] 時間的特徴量を生成中...


INFO:11_ensemble_gbdt_advanced_features:時間的特徴量を生成中...


[2026-08-08 08:11:32] [INFO] 交互作用特徴量を生成中...


INFO:11_ensemble_gbdt_advanced_features:交互作用特徴量を生成中...


[2026-08-08 08:11:32] [INFO] 特徴量処理完了


INFO:11_ensemble_gbdt_advanced_features:特徴量処理完了


In [13]:
logger.info("-" * 60)
logger.info("特徴量の統合")
logger.info("-" * 60)

train_persona_features = train_persona.drop(columns=[TARGET_COL])

# 月次集約特徴量を統合
train_features = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
train_features = train_features.merge(train_cat_change, on=ID_COL, how="left")
train_features = train_features.merge(train_missing, on=ID_COL, how="left")
train_features = train_features.merge(train_domain, on=ID_COL, how="left")
train_features = train_features.merge(train_advanced_stats, on=ID_COL, how="left")
train_features = train_features.merge(train_cluster, on=ID_COL, how="left")

test_features = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
test_features = test_features.merge(test_cat_change, on=ID_COL, how="left")
test_features = test_features.merge(test_missing, on=ID_COL, how="left")
test_features = test_features.merge(test_domain, on=ID_COL, how="left")
test_features = test_features.merge(test_advanced_stats, on=ID_COL, how="left")
test_features = test_features.merge(test_cluster, on=ID_COL, how="left")

logger.info(f"Train: {train_features.shape}, Test: {test_features.shape}")

# カテゴリカル変数の処理
cat_cols = ["入社区分", "性別", "最終学歴", "専攻分野", "採用経路", "初期職種", "初期勤務地", "初期役割"]
for col in cat_cols:
    if col in train_features.columns:
        le = LabelEncoder()
        combined = pd.concat([train_features[col].fillna("missing"), test_features[col].fillna("missing")])
        le.fit(combined)
        train_features[col] = le.transform(train_features[col].fillna("missing"))
        test_features[col] = le.transform(test_features[col].fillna("missing"))

# 不要な列の削除
drop_cols = [ID_COL, "入社日", "入社時メモ", "上司からのフィードバック", "同僚からのフィードバック", "初期部署ID", "初期等級", "最終学歴", "前職職種"]
drop_cols_exist = [col for col in drop_cols if col in train_features.columns]
train_features = train_features.drop(columns=drop_cols_exist)
test_features = test_features.drop(columns=drop_cols_exist)

# NaNを-999で埋める
X_train = train_features.fillna(-999)
y_train_target = y_train.copy()
X_test = test_features.fillna(-999)

logger.info(f"X_train Shape: {X_train.shape}, X_test Shape: {X_test.shape}")
logger.info(f"最終特徴量数: {X_train.shape[1]}")

[2026-08-08 08:11:32] [INFO] ------------------------------------------------------------


INFO:11_ensemble_gbdt_advanced_features:------------------------------------------------------------


[2026-08-08 08:11:32] [INFO] 特徴量の統合


INFO:11_ensemble_gbdt_advanced_features:特徴量の統合


[2026-08-08 08:11:32] [INFO] ------------------------------------------------------------


...

INFO:11_ensemble_gbdt_advanced_features:Train: (2761, 284), Test: (2502, 284)


[2026-08-08 08:11:32] [INFO] X_train Shape: (2761, 275), X_test Shape: (2502, 275)


INFO:11_ensemble_gbdt_advanced_features:X_train Shape: (2761, 275), X_test Shape: (2502, 275)


[2026-08-08 08:11:32] [INFO] 最終特徴量数: 275


INFO:11_ensemble_gbdt_advanced_features:最終特徴量数: 275


## 4. 個別モデルの学習

CatBoost, LightGBM, XGBoostの3モデルを各v2で学習します。

In [14]:
import optuna

logger.info("=" * 60)
logger.info("CatBoostモデルの学習 (1回目)")
logger.info("=" * 60)

# sort_colとして入社日を使用
train_features_with_date = train_persona[[ID_COL, '入社日']].merge(
    pd.DataFrame({ID_COL: train_ids}), on=ID_COL, how='right'
)
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

input_data_cat = {
    "X_train": X_train_with_sort,
    "y_train": y_train_target,
    "X_test": X_test,
    "sort_col": "入社日",
}

cat_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR / "catboost"),
    "cv_strategy": "timeseries",
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3,
    "border_count": 128,
    "bagging_temperature": 0.2,
    "random_strength": 1,
    "early_stopping_rounds": 50,
    "verbose": False,
    "task_type": "GPU",  # GPU対応
}

logger.info("CatBoostトレーニング開始 (1回目)...")
result_cat, _ = run_cat(input_data_cat, cat_params)

# ==========================================
# 特徴量重要度を計測し、不要な特徴量を削除
# ==========================================
if "models" in result_cat and len(result_cat["models"]) > 0:
    logger.info("CatBoostの特徴量重要度を計測し、重要度0の特徴量を削除します")
    cat_models = result_cat["models"]

    feature_names = cat_models[0].feature_names_
    feature_importance_sum = np.zeros(len(feature_names))

    for model in cat_models:
        feature_importance_sum += model.get_feature_importance()

    feature_importance_avg = feature_importance_sum / len(cat_models)

    # 重要度が0より大きい特徴量のみ保持
    important_features = [feat for feat, imp in zip(feature_names, feature_importance_avg) if imp > 0]

    logger.info(f"全{len(feature_names)}特徴量中、重要度が0の{len(feature_names) - len(important_features)}個を削除")

    # データセットの更新
    input_data_cat["X_train"] = X_train_with_sort[important_features + ["入社日"]]
    input_data_cat["X_test"] = X_test[important_features]

    # ==========================================
    # Optunaによるハイパーパラメータ探索 (GPU)
    # ==========================================
    logger.info("CatBoostのOptunaモジュールによる探索開始...")

    cat_params["n_trials"] = 10
    _, best_params = run_cat_optuna(input_data_cat, cat_params)
    logger.info(f"Best CatBoost Params: {best_params}")

    # ベストパラメータで深い再学習
    logger.info("CatBoostベストパラメータでの深い再学習開始...")
    cat_params.update(best_params)
    cat_params["iterations"] = 2000
    cat_params["early_stopping_rounds"] = 100
    result_cat, _ = run_cat(input_data_cat, cat_params)

cat_oof_score = result_cat["oof_score"]
cat_test_preds = result_cat["test_preds"]
cat_oof_preds = result_cat["oof_preds"]
logger.info(f"CatBoost OOF Score (LogLoss): {cat_oof_score:.6f}")


[2026-08-08 08:11:32] [INFO] ============================================================


INFO:11_ensemble_gbdt_advanced_features:============================================================


[2026-08-08 08:11:33] [INFO] CatBoostモデルの学習 (1回目)


INFO:11_ensemble_gbdt_advanced_features:CatBoostモデルの学習 (1回目)


[2026-08-08 08:11:33] [INFO] ============================================================


...

INFO:11_ensemble_gbdt_advanced_features:Best CatBoost Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/content/drive/MyDrive/jaggle_2026/saved_models/20260808/11_ensemble_gbdt_advanced_features/catboost', 'cv_strategy': 'timeseries', 'iterations': 1000, 'learning_rate': 0.015957084694148364, 'depth': 3, 'l2_leaf_reg': 2.9154431891537547, 'border_count': 128, 'bagging_temperature': 0.7080725777960455, 'random_strength': 0.001026006512489678, 'early_stopping_rounds': 50, 'verbose': False, 'task_type': 'GPU', 'n_trials': 10}


[2026-08-08 08:26:54] [INFO] CatBoostベストパラメータでの深い再学習開始...


INFO:11_ensemble_gbdt_advanced_features:CatBoostベストパラメータでの深い再学習開始...


[2026-08-08 08:27:35] [INFO] CatBoost OOF Score (LogLoss): 0.586817


INFO:11_ensemble_gbdt_advanced_features:CatBoost OOF Score (LogLoss): 0.586817


In [15]:
logger.info("=" * 60)
logger.info("LightGBMモデルの学習 (1回目)")
logger.info("=" * 60)

input_data_lgb = {
    "X_train": X_train_with_sort,
    "y_train": y_train_target,
    "X_test": X_test,
    "sort_col": "入社日",
}

lgb_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR / "lightgbm"),
    "objective": "binary",
    "metric": "binary_logloss",
    "early_stopping_rounds": 50,
    "verbose_eval": False,
    "cv_strategy": "timeseries",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_child_samples": 20,
    "verbose": -1,
    "n_estimators": 1000,
    "device": "gpu",  # GPU対応
}

logger.info("LightGBMトレーニング開始 (1回目)...")
result_lgb, _ = run_lgb(input_data_lgb, lgb_params)

# ==========================================
# 特徴量重要度を計測し、不要な特徴量を削除
# ==========================================
if "models" in result_lgb and len(result_lgb["models"]) > 0:
    logger.info("LightGBMの特徴量重要度を計測し、重要度0の特徴量を削除します")
    lgb_models = result_lgb["models"]

    # 全Foldの平均重要度を計算
    feature_names = lgb_models[0].feature_name()
    feature_importance_sum = np.zeros(len(feature_names))

    for model in lgb_models:
        feature_importance_sum += model.feature_importance(importance_type='gain')

    feature_importance_avg = feature_importance_sum / len(lgb_models)

    # 重要度が0より大きい特徴量のみ保持
    important_features = [feat for feat, imp in zip(feature_names, feature_importance_avg) if imp > 0]

    logger.info(f"全{len(feature_names)}特徴量中、重要度が0の{len(feature_names) - len(important_features)}個を削除")

    # データセットの更新
    input_data_lgb["X_train"] = X_train_with_sort[important_features + ["入社日"]]
    input_data_lgb["X_test"] = X_test[important_features]

    # ==========================================
    # Optunaによるハイパーパラメータ探索 (GPU)
    # ==========================================
    logger.info("LightGBMのOptunaモジュールによる探索開始...")

    lgb_params["n_trials"] = 10
    _, best_params = run_lgb_optuna(input_data_lgb, lgb_params)
    logger.info(f"Best LightGBM Params: {best_params}")

    # ベストパラメータで深い再学習
    logger.info("LightGBMベストパラメータでの深い再学習開始...")
    lgb_params.update(best_params)
    lgb_params["n_estimators"] = 2000
    lgb_params["early_stopping_rounds"] = 100
    result_lgb, _ = run_lgb(input_data_lgb, lgb_params)

lgb_oof_score = result_lgb["oof_score"]
lgb_test_preds = result_lgb["test_preds"]
lgb_oof_preds = result_lgb["oof_preds"]
logger.info(f"LightGBM OOF Score (LogLoss): {lgb_oof_score:.6f}")


[2026-08-08 08:27:35] [INFO] ============================================================


INFO:11_ensemble_gbdt_advanced_features:============================================================


[2026-08-08 08:27:35] [INFO] LightGBMモデルの学習 (1回目)


INFO:11_ensemble_gbdt_advanced_features:LightGBMモデルの学習 (1回目)


[2026-08-08 08:27:35] [INFO] ============================================================


...

INFO:11_ensemble_gbdt_advanced_features:Best LightGBM Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/content/drive/MyDrive/jaggle_2026/saved_models/20260808/11_ensemble_gbdt_advanced_features/lightgbm', 'objective': 'binary', 'metric': 'binary_logloss', 'early_stopping_rounds': 50, 'verbose_eval': False, 'cv_strategy': 'timeseries', 'boosting_type': 'gbdt', 'num_leaves': 37, 'learning_rate': 0.01303561122512888, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 36, 'verbose': -1, 'n_estimators': 1000, 'device': 'gpu', 'n_trials': 10, 'max_depth': 3, 'subsample': 0.6943386448447411, 'colsample_bytree': 0.6356745158869479, 'reg_alpha': 0.28749982347407854, 'reg_lambda': 1.6247252885719427e-05}


[2026-08-08 08:28:18] [INFO] LightGBMベストパラメータでの深い再学習開始...


INFO:11_ensemble_gbdt_advanced_features:LightGBMベストパラメータでの深い再学習開始...


[2026-08-08 08:28:23] [INFO] LightGBM OOF Score (LogLoss): 0.587679


INFO:11_ensemble_gbdt_advanced_features:LightGBM OOF Score (LogLoss): 0.587679


In [16]:
logger.info("=" * 60)
logger.info("XGBoostモデルの学習 (1回目)")
logger.info("=" * 60)

input_data_xgb = {
    "X_train": X_train_with_sort,
    "y_train": y_train_target,
    "X_test": X_test,
    "sort_col": "入社日",
}

xgb_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR / "xgboost"),
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "cv_strategy": "timeseries",
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "n_estimators": 1000,
    "tree_method": "hist",
    "device": "cuda",  # XGBoost 2.0以降のGPU対応
    "verbose": False,
}

logger.info("XGBoostトレーニング開始 (1回目)...")
result_xgb, _ = run_xgb(input_data_xgb, xgb_params)

# ==========================================
# 特徴量重要度を計測し、不要な特徴量を削除
# ==========================================
if "models" in result_xgb and len(result_xgb["models"]) > 0:
    logger.info("XGBoostの特徴量重要度を計測し、重要度0の特徴量を削除します")
    xgb_models = result_xgb["models"]

    feature_names = xgb_models[0].feature_names
    feat_imp_dict = {feat: 0.0 for feat in feature_names}

    for model in xgb_models:
        # XGBoostのBoosterからgainを取得
        score = model.get_score(importance_type='gain')
        for k, v in score.items():
            feat_imp_dict[k] += v

    # 重要度が0より大きい特徴量のみ保持
    important_features = [feat for feat, imp in feat_imp_dict.items() if imp > 0]

    logger.info(f"全{len(feature_names)}特徴量中、重要度が0の{len(feature_names) - len(important_features)}個を削除")

    # データセットの更新
    input_data_xgb["X_train"] = X_train_with_sort[important_features + ["入社日"]]
    input_data_xgb["X_test"] = X_test[important_features]

    # ==========================================
    # Optunaによるハイパーパラメータ探索 (GPU)
    # ==========================================
    logger.info("XGBoostのOptunaモジュールによる探索開始...")

    xgb_params["n_trials"] = 10
    _, best_params = run_xgb_optuna(input_data_xgb, xgb_params)
    logger.info(f"Best XGBoost Params: {best_params}")

    # ベストパラメータで深い再学習
    logger.info("XGBoostベストパラメータでの深い再学習開始...")
    xgb_params.update(best_params)
    xgb_params["n_estimators"] = 2000
    xgb_params["early_stopping_rounds"] = 100
    result_xgb, _ = run_xgb(input_data_xgb, xgb_params)

xgb_oof_score = result_xgb["oof_score"]
xgb_test_preds = result_xgb["test_preds"]
xgb_oof_preds = result_xgb["oof_preds"]
logger.info(f"XGBoost OOF Score (LogLoss): {xgb_oof_score:.6f}")


[2026-08-08 08:28:23] [INFO] ============================================================


INFO:11_ensemble_gbdt_advanced_features:============================================================


[2026-08-08 08:28:23] [INFO] XGBoostモデルの学習 (1回目)


INFO:11_ensemble_gbdt_advanced_features:XGBoostモデルの学習 (1回目)


[2026-08-08 08:28:23] [INFO] ============================================================


...

INFO:11_ensemble_gbdt_advanced_features:Best XGBoost Params: {'n_splits': 5, 'seed': 42, 'save_dir': '/content/drive/MyDrive/jaggle_2026/saved_models/20260808/11_ensemble_gbdt_advanced_features/xgboost', 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'early_stopping_rounds': 50, 'cv_strategy': 'timeseries', 'max_depth': 5, 'learning_rate': 0.11265466963346032, 'subsample': 0.8421165132560784, 'colsample_bytree': 0.7200762468698007, 'min_child_weight': 2, 'n_estimators': 1000, 'tree_method': 'hist', 'device': 'cuda', 'verbose': False, 'n_trials': 10, 'alpha': 1.254134495897175e-07, 'lambda': 0.00028614897264046574}


[2026-08-08 08:28:31] [INFO] XGBoostベストパラメータでの深い再学習開始...


INFO:11_ensemble_gbdt_advanced_features:XGBoostベストパラメータでの深い再学習開始...


[2026-08-08 08:28:32] [INFO] XGBoost OOF Score (LogLoss): 0.611024


INFO:11_ensemble_gbdt_advanced_features:XGBoost OOF Score (LogLoss): 0.611024


## 5. アンサンブル手法

3つのモデルを組み合わせて、さらに高い性能を目指します。

1. **Simple Average** - 3モデルの単純平均
2. **Weighted Average** - OOFスコアベースの重み付き平均
3. **Stacking** - メタモデル（Logistic Regression）

In [17]:
logger.info("=" * 60)
logger.info("アンサンブル")
logger.info("=" * 60)

# モデル内部で '入社日' でソートされているため、ターゲットも同様にソートする
sort_idx = X_train_with_sort.sort_values('入社日').index
y_train_sorted = y_train_target.iloc[sort_idx].reset_index(drop=True)

# TimeSeriesSplitの仕様上、最初の学習データ部分にはOOF予測が存在せず0.0となるため、
# 有効な予測が行われたインデックスのみを抽出する
valid_idx = cat_oof_preds > 0
y_valid = y_train_sorted[valid_idx]

# 1. Simple Average
logger.info("1. Simple Average")
simple_avg_test = (cat_test_preds + lgb_test_preds + xgb_test_preds) / 3
simple_avg_oof = (cat_oof_preds + lgb_oof_preds + xgb_oof_preds) / 3
simple_avg_oof_score = calculate_logloss(y_valid, simple_avg_oof[valid_idx])
logger.info(f"Simple Average OOF Score: {simple_avg_oof_score:.6f}")

# 2. Weighted Average (スコアの逆数を重みとする)
logger.info("2. Weighted Average")
scores = np.array([cat_oof_score, lgb_oof_score, xgb_oof_score])
weights = (1 / scores) / (1 / scores).sum()  # スコアが低いほど重みが大きい
logger.info(f"Weights: CatBoost={weights[0]:.4f}, LightGBM={weights[1]:.4f}, XGBoost={weights[2]:.4f}")

weighted_avg_test = cat_test_preds * weights[0] + lgb_test_preds * weights[1] + xgb_test_preds * weights[2]
weighted_avg_oof = cat_oof_preds * weights[0] + lgb_oof_preds * weights[1] + xgb_oof_preds * weights[2]
weighted_avg_oof_score = calculate_logloss(y_valid, weighted_avg_oof[valid_idx])
logger.info(f"Weighted Average OOF Score: {weighted_avg_oof_score:.6f}")

# 3. Stacking
logger.info("3. Stacking with Logistic Regression")
oof_stack = np.column_stack([cat_oof_preds, lgb_oof_preds, xgb_oof_preds])
test_stack = np.column_stack([cat_test_preds, lgb_test_preds, xgb_test_preds])

# メタモデルの学習も有効なOOF予測のみで行う
meta_model = LogisticRegression(random_state=SEED, max_iter=1000)
meta_model.fit(oof_stack[valid_idx], y_valid)
stacking_test = meta_model.predict_proba(test_stack)[:, 1]
stacking_oof_valid = meta_model.predict_proba(oof_stack[valid_idx])[:, 1]

# フルサイズのoof予測を作成（後続の処理用）
stacking_oof = np.zeros_like(cat_oof_preds)
stacking_oof[valid_idx] = stacking_oof_valid

stacking_oof_score = calculate_logloss(y_valid, stacking_oof_valid)
logger.info(f"Stacking OOF Score: {stacking_oof_score:.6f}")
logger.info(f"Meta model coefs: CatBoost={meta_model.coef_[0][0]:.4f}, LightGBM={meta_model.coef_[0][1]:.4f}, XGBoost={meta_model.coef_[0][2]:.4f}")

# スコアサマリ
logger.info("=" * 60)
logger.info("スコアサマリ")
logger.info("=" * 60)
logger.info(f"CatBoost OOF Score:        {cat_oof_score:.6f}")
logger.info(f"LightGBM OOF Score:        {lgb_oof_score:.6f}")
logger.info(f"XGBoost OOF Score:         {xgb_oof_score:.6f}")
logger.info(f"Simple Average OOF Score:  {simple_avg_oof_score:.6f}")
logger.info(f"Weighted Average OOF Score:{weighted_avg_oof_score:.6f}")
logger.info(f"Stacking OOF Score:        {stacking_oof_score:.6f}")
logger.info("=" * 60)


[2026-08-08 08:28:33] [INFO] ============================================================


INFO:11_ensemble_gbdt_advanced_features:============================================================


[2026-08-08 08:28:33] [INFO] アンサンブル


INFO:11_ensemble_gbdt_advanced_features:アンサンブル


[2026-08-08 08:28:33] [INFO] ============================================================


...

INFO:11_ensemble_gbdt_advanced_features:Weighted Average OOF Score:0.590245


[2026-08-08 08:28:33] [INFO] Stacking OOF Score:        0.585394


INFO:11_ensemble_gbdt_advanced_features:Stacking OOF Score:        0.585394


[2026-08-08 08:28:33] [INFO] ============================================================


INFO:11_ensemble_gbdt_advanced_features:============================================================


In [18]:
logger.info("-" * 60)
logger.info("結果の保存")
logger.info("-" * 60)

# Simple Average
sub_simple = pd.DataFrame({ID_COL: test_ids, TARGET_COL: simple_avg_test})
sub_simple.to_csv(SUBMISSION_SIMPLE_PATH, index=False, header=False)
logger.info(f"Simple Average保存: {SUBMISSION_SIMPLE_PATH}")

# Weighted Average
sub_weighted = pd.DataFrame({ID_COL: test_ids, TARGET_COL: weighted_avg_test})
sub_weighted.to_csv(SUBMISSION_WEIGHTED_PATH, index=False, header=False)
logger.info(f"Weighted Average保存: {SUBMISSION_WEIGHTED_PATH}")

# Stacking
sub_stacking = pd.DataFrame({ID_COL: test_ids, TARGET_COL: stacking_test})
sub_stacking.to_csv(SUBMISSION_STACKING_PATH, index=False, header=False)
logger.info(f"Stacking保存: {SUBMISSION_STACKING_PATH}")

logger.info("=" * 60)
logger.info("=== 実験完了 ===")
logger.info("=" * 60)

print(f"\n■ 個別モデルOOF Scores:")
print(f"  CatBoost:  {cat_oof_score:.6f}")
print(f"  LightGBM:  {lgb_oof_score:.6f}")
print(f"  XGBoost:   {xgb_oof_score:.6f}")
print(f"\n■ アンサンブルOOF Scores (修正後):")
print(f"  Simple Avg:    {simple_avg_oof_score:.6f}")
print(f"  Weighted Avg:  {weighted_avg_oof_score:.6f}")
print(f"  Stacking:      {stacking_oof_score:.6f}")
print(f"\n■ 保存ファイル:")
print(f"  Simple Average:  {SUBMISSION_SIMPLE_PATH}")
print(f"  Weighted Average: {SUBMISSION_WEIGHTED_PATH}")
print(f"  Stacking:        {SUBMISSION_STACKING_PATH}")

[2026-08-08 08:28:33] [INFO] ------------------------------------------------------------


INFO:11_ensemble_gbdt_advanced_features:------------------------------------------------------------


[2026-08-08 08:28:33] [INFO] 結果の保存


INFO:11_ensemble_gbdt_advanced_features:結果の保存


[2026-08-08 08:28:33] [INFO] ------------------------------------------------------------


...

[2026-08-08 08:28:33] [INFO] === 実験完了 ===


INFO:11_ensemble_gbdt_advanced_features:=== 実験完了 ===


[2026-08-08 08:28:33] [INFO] ============================================================


INFO:11_ensemble_gbdt_advanced_features:============================================================



■ 個別モデルOOF Scores:
  CatBoost:  0.586817
  LightGBM:  0.587679
  XGBoost:   0.611024

■ アンサンブルOOF Scores (修正後):
  Simple Avg:    0.590424
  Weighted Avg:  0.590245
  Stacking:      0.585394

■ 保存ファイル:
  Simple Average:  /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_11_ensemble_gbdt_advanced_features_simple_avg.csv
  Weighted Average: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_11_ensemble_gbdt_advanced_features_weighted_avg.csv
  Stacking:        /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_11_ensemble_gbdt_advanced_features_stacking.csv


## 🎉 完了

このノートブックでは、以下を実装しました：

### 特徴量エンジニアリング
1. ✅ **既存特徴量** - 月次集約、カテゴリ変化、欠損値、ドメイン知識
2. ✅ **高度な特徴量** - 統計的特徴、クラスター、多項式、比率

### モデリング
3. ✅ **CatBoost v2** - catboost_model_v2.py使用
4. ✅ **LightGBM v2** - lgbm_model_v2.py使用
5. ✅ **XGBoost v2** - xgboost_model_v2.py使用
6. ✅ **TimeSeriesSplit** - 入社日順でCV

### アンサンブル
7. ✅ **Simple Average** - 3モデルの平均
8. ✅ **Weighted Average** - OOFスコアベースの重み
9. ✅ **Stacking** - Logistic Regressionメタモデル

### 次のステップ
- 各アンサンブル結果をKaggle/コンペに提出
- Public Scoreを確認し、最も高いスコアの手法を採用
- 必要に応じてハイパーパラメータ最適化（Optuna）